# Importación de librerías

In [ ]:
# Módulo
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

In [ ]:
# Para que se puedan utilizar funciones desde el notebook
from src.utils.files import read_file
from src.utils.config import new_data_prices,prices, load_env_file
load_env_file()

In [ ]:
use_minio = False
minio = {"minio_write": False, "minio_read": use_minio}

# Carga de datos

In [ ]:
df_datos = read_file(prices, minio)
df_nuevo = read_file(new_data_prices, minio)

df_datos['release_year'] = df_datos['release_year'].astype(int)
df_nuevo['release_year'] = df_nuevo['release_year'].astype(int)

df_cleaned_datos = df_datos.drop(columns=["id", "name", "v_resnet", "v_clip", "v_convnext"], errors='ignore').copy()
df_cleaned_nuevo = df_nuevo.drop(columns=["id", "name", "v_resnet", "v_clip", "v_convnext"], errors='ignore').copy()

## Distribución de la variable respuesta

In [ ]:
fig = make_subplots(rows=1,cols=2, specs= [[{"type": "domain"}, {"type": "domain"}]], subplot_titles= ("Datos extraídos", "Datos nuevos"))

orden = [
    "[0.01,4.99]",
    "[5.00,9.99]",
    "[10.00,14.99]",
    "[15.00,19.99]",
    "[20.00,29.99]",
    "[30.00,39.99]",
    ">40"
]

values_df_datos = df_cleaned_datos["price_range"].value_counts()
values_df_datos = values_df_datos.reindex(orden)

values_df_nuevo = df_cleaned_nuevo["price_range"].value_counts()
values_df_nuevo = values_df_nuevo.reindex(orden)


fig.add_trace(go.Pie(
    values=values_df_datos.values, 
    labels=values_df_datos.index, 
    sort=False,
    marker=dict(colors=["lightcoral", "teal", "orange", "skyblue", "mediumpurple", "gold", "lightgreen"])
), row=1,col=1)


fig.add_trace(go.Pie(
    values=values_df_nuevo.values, 
    labels=values_df_nuevo.index, 
    sort=False,
    marker=dict(colors=["lightcoral", "teal", "orange", "skyblue", "mediumpurple", "gold", "lightgreen"])
), row=1,col=2)

fig.update_layout(
    width=800,
    height=700,
    title={
        "text": "<b>Proporción de los rangos de precio</b>",
        "x": 0.5,
        "y": 0.9,
        "font": dict(size=24)
    },
    paper_bgcolor="white",
    plot_bgcolor="whitesmoke",
    font=dict(
        family="Times New Roman",
        color="black"
    ),
    legend_title = "Rangos de precios",
    annotations=[
        dict(y=0.85, x=0.22, text="Datos extraídos", showarrow=False),
        dict(y=0.85, x=0.78, text="Datos nuevos", showarrow=False)
    ]
)

fig.show()

Las distribuciones de los rangos de precios son bastante similares, el orden relativo de frecuencia para cada género se mantiene. La única diferencia significativa es el incremento del número de juegos en el rango de precio [0.01, 4.99] y la reducción del porcentaje en los rangos a partir de los 10 euros. Esta diferencia puede deberse al tamaño del conjunto de nuevos datos extraídos (898 juegos), que es muy pequeño en comparación a los datos que se tomaron para este problema originalmente (20000 juegos).

## Comparación de distribución de las variables más importantes

### Correlaciones

In [ ]:
continuas = [col for col in df_cleaned_datos.columns if df_cleaned_datos[col].nunique() > 2 and col not in ['price_overview', 'price_range']]
correlaciones = df_cleaned_datos.drop(columns=["price_range", "price_overview"]).corrwith(
    df_cleaned_datos['price_overview'], method='spearman').sort_values()

fig1 = go.Figure(go.Bar(
    x=correlaciones.values,
    y=correlaciones.index,
    orientation='h',
    marker_color=['#A8754C' if x < 0 else "#187FF5" for x in correlaciones.values]
))

fig1.add_vline(x=0, line_width=1, line_color='black')

fig1.update_layout(
    width=1100,
    height=800,
    title=dict(
        text="<b>Correlación con price_overview</b>",
        font=dict(size=24)
    ),
    title_x=0.5,
    title_xanchor='center',
    plot_bgcolor="whitesmoke",
    paper_bgcolor="white",
    font=dict(
        family="Times New Roman",
        color="black"
    )
)

fig1.update_xaxes(
    title_text="Coeficiente de Correlación (Spearman)",
    title_font=dict(size=18),
    gridcolor="lightgrey"
)

fig1.update_yaxes(
    title_text="Variables",
    title_font=dict(size=18),
    gridcolor="lightgrey",
    tickfont=dict(size=10)
)

fig1.show()

resultados = {}
for col in continuas:
    r, p = spearmanr(df_cleaned_datos[col], df_cleaned_datos['price_overview'])
    resultados[col] = {'r (coeficiente de Spearman)': r, 'p-valor': p}

Las 3 variables con mayor correlación absoluta son: `ema_precio_publishers`, `max_historico_precio_publishers`, `Steam Cloud`.

Pasamos ahora a comparar su distribución con los nuevos datos para comprobar si existen variaciones significativas.

In [ ]:
columnas = ["ema_precio_publishers", "max_historico_precio_publishers"]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "ema_precio_publishers (Datos existentes)",
        "ema_precio_publishers (Datos nuevos)",
        "max_historico_precio_publishers (Datos existentes)",
        "max_historico_precio_publishers (Datos nuevos)"
    )
)

color_1 = "#A8754C"
color_2 = "#187FF5"

for i, col in enumerate(columnas, start=1):

    fig.add_trace(
        go.Box(
            y=df_cleaned_datos[col],
            marker_color=color_1,
            marker_line=dict(color="white", width=1),
            name="Existentes"
        ),
        row=i, col=1
    )

    fig.add_trace(
        go.Box(
            y=df_cleaned_nuevo[col],
            marker_color=color_2,
            marker_line=dict(color="white", width=1),
            name="Nuevos"
        ),
        row=i, col=2
    )



fig.update_layout(
    height=700,
    width=1000,
    title=dict(
        text="Comparación de distribuciones",
        x=0.5,
        font=dict(size=22)
    ),
    template="plotly_white",
    showlegend=False,
    bargap=0.1,
    paper_bgcolor="white",
    plot_bgcolor="whitesmoke",
    font=dict(
        family="Times New Roman",
        color="black"
    ),
    legend_title = "Rangos de precios",
)

fig.show()

Las distribuciones para ambas variables son muy similares entre los datos existentes y los datos nuevos, se aprecia que la única diferencia relevante es la cantidad de juegos en ambos datasets.

In [ ]:
print(f"Porcentaje de juegos que permiten Steam Cloud en los datos existentes: {(df_cleaned_datos["Steam Cloud"].sum() / len(df_cleaned_datos))*100:3f}%")
print(f"Porcentaje de juegos que permiten Steam Cloud en los datos nuevos: {(df_cleaned_nuevo["Steam Cloud"].sum() / len(df_cleaned_nuevo))*100:3f}%")

Se aprecia que el porcentaje varía ligeramente en ambas pruebas y esto puede deberse al tamaño reducido del conjunto de nuevos datos extraídos. La diferencia no es significativa y se puede asumir que el porcentaje sigue siendo muy similar.